# Dataset 6 — Individual Household Electric Power Consumption (UCI)
## Etapa B — Python / Pandas

Pré-requisito: Etapa A no Orange — atenção que aqui o dataset é carregado **original** (não a amostra usada em aula), com tratamento de ausentes feito no Orange antes de amostrar. Select Columns mantendo `Global_active_power`, `Global_reactive_power`, `Voltage`, `Global_intensity` e os três `Sub_metering`; nova amostra aleatória de 10% dos dados tratados; exportar CSV. Ajuste `CAMINHO_CSV` para o arquivo exportado.

In [ ]:
import pandas as pd

CAMINHO_CSV = "amostra_household_power.csv"  # ajustar para o nome/caminho real do arquivo exportado do Orange

df = pd.read_csv(CAMINHO_CSV)
df.columns

### 0. Conferência de ausentes

O tratamento principal de ausentes é feito na Etapa A (Orange), mas vale reconferir aqui — se a amostra ainda tiver ausentes, isso indica que o tratamento anterior não foi completo.

In [ ]:
df.isnull().sum()

### 1. Organizar atributos com nomes mais simples

In [ ]:
df = df.rename(columns={
    "Global_active_power": "Pot_Ativa",
    "Global_reactive_power": "Pot_Reativa",
    "Voltage": "Tensao",
    "Global_intensity": "Corrente",
    "Sub_metering_1": "Submedicao_1",
    "Sub_metering_2": "Submedicao_2",
    "Sub_metering_3": "Submedicao_3",
})
df.head()

### 2. Valor máximo de potência ativa

In [ ]:
max_pot_ativa = df["Pot_Ativa"].max()
max_pot_ativa

### 3. Limiar de 75% do máximo e DataFrame acima do limite

In [ ]:
limiar_75 = 0.75 * max_pot_ativa
df_pot_alta = df[df["Pot_Ativa"] > limiar_75]
df_pot_alta.head()

### 4. Quantidade e percentual de registros selecionados

In [ ]:
qtd_pot_alta = len(df_pot_alta)
percentual_pot_alta = qtd_pot_alta / len(df) * 100

print(f"Registros de potência ativa elevada: {qtd_pot_alta}")
print(f"Percentual sobre o total da amostra: {percentual_pot_alta:.2f}%")

### 5. Corrente média da amostra

In [ ]:
corrente_media = df["Corrente"].mean()
corrente_media

### 6. Segundo DataFrame: potência ativa acima de 75% do máximo E corrente acima da média

In [ ]:
df_pot_corrente_alta = df[
    (df["Pot_Ativa"] > limiar_75) &
    (df["Corrente"] > corrente_media)
]

qtd_pot_corrente_alta = len(df_pot_corrente_alta)
percentual_pot_corrente_alta = qtd_pot_corrente_alta / len(df) * 100

print(f"Registros com potência ativa alta E corrente acima da média: {qtd_pot_corrente_alta}")
print(f"Percentual sobre o total da amostra: {percentual_pot_corrente_alta:.2f}%")

### 7. Comparação entre os dois conjuntos

In [ ]:
print(f"Só potência ativa alta: {qtd_pot_alta} registros")
print(f"Potência ativa alta E corrente acima da média: {qtd_pot_corrente_alta} registros")
print(f"Redução ao adicionar a corrente como segunda condição: {qtd_pot_alta - qtd_pot_corrente_alta} registros a menos")

**Interpretação (preencher com os valores impressos acima antes de entregar):**

Potência ativa (`P = V × I × cos φ`) e corrente estão fisicamente relacionadas, então é esperado que boa parte dos registros de potência elevada também tenha corrente acima da média — a questão é o quanto a interseção se aproxima do conjunto original.

- Se a queda de `<preencher qtd_pot_alta>` para `<preencher qtd_pot_corrente_alta>` for pequena, os dois critérios são quase redundantes nesta amostra — a corrente já "carrega" a mesma informação da potência ativa, e adicionar o segundo filtro pouco refina a seleção.
- Se a queda for grande, existem episódios de potência ativa alta com corrente relativamente baixa — compatível com tensão mais alta que o normal, ou fator de potência elevado sustentando P alto sem I proporcionalmente alto. Esses casos residuais é que valeriam investigação separada, já que fogem do padrão simples de "mais corrente, mais potência".